===========================================================  
**Title: Single-Cell Paired BCR Clonotype Assignment**

**POC: Dr. Ian York**  
Team Lead, Molecular Virology and Vaccines Team  
Immunology and Pathogenesis Branch   
Influenza Division, NCIRD, OID   
Centers for Disease Control and Prevention   
**Email:** ite1@cdc.gov  

Other Contacts: Sujatha Seenu  
**Email:** iun1@cdc.gov  
============================================================
#### Overview

This workflow identifies clonotypes from single-cell BCR repertoire data using paired heavy- and light-chain information. Cells are grouped according to heavy-chain and light-chain V gene usage, J gene usage, and CDR3 amino acid sequence length. Clonotypes are inferred using a network-based clustering approach, where cells with paired BCR sequences sharing **≥80% CDR3 amino acid sequence identity** are assigned to the same clonotype.

##### Required Column Naming Convention: 
Heavy-chain columns must be prefixed with **IGH_**, and light-chain columns must be prefixed with **IGL_**.

In [1]:
### Python libraries
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path
from Bio import SeqIO, Seq
import itertools
from itertools import combinations

### Import or Install networkx python library
import networkx as nx  ### pip install networkx



In [2]:
#Input Validation and Data Loading
def load_input_data(prepared_dataframe=None, in_file_path=None, out_file_path=None):
 
    if out_file_path is not None:
        out_file_path = Path(out_file_path)
        if not out_file_path.parent.exists():
            raise FileNotFoundError(f'Output directory does not exist: {out_file_path.parent}')
 
    if prepared_dataframe is not None:
        if not isinstance(prepared_dataframe, pd.DataFrame):
            raise TypeError("'prepared_dataframe' must be a pandas DataFrame")
        return prepared_dataframe.copy()
 
    if in_file_path is None:
        raise ValueError("Either 'prepared_dataframe' or 'in_file_path' must be provided")
 
    in_file_path = Path(in_file_path)
    if not in_file_path.exists():
        raise FileNotFoundError(f'Input file not found: {in_file_path}')
 
    suffix = in_file_path.suffix.lower()
    if suffix == '.tsv':
        return pd.read_csv(in_file_path, sep='\t', low_memory=False)
    
    if suffix == '.xlsx':
        return pd.read_excel(in_file_path)
 
    raise ValueError("File extension isn't .tsv, .csv, or .xlsx")


def check_input(df):
    
    # Required column names
    req_fields = ['IGH_cdr3_aa', 'IGH_v_call', 'IGH_j_call','IGL_cdr3_aa', 'IGL_v_call', 'IGL_j_call' ] 
    cols = df.columns
 
    missing = [field for field in req_fields if field not in cols]
    # Adjust required fields when junction_aa is provided instead of cdr3_aa for heavy and light chains
    if 'IGH_cdr3_aa' in missing and 'IGH_junction_aa' in df.columns: 
        missing.remove("IGH_cdr3_aa")
        req_fields.remove("IGH_cdr3_aa")
    if 'IGL_cdr3_aa' in missing and 'IGL_junction_aa' in df.columns: 
        missing.remove("IGL_cdr3_aa")
        req_fields.remove("IGL_cdr3_aa")
    if missing:
        raise ValueError('Missing required columns: %s' % ', '.join(missing))
                 
    # Need either junction_aa or cdr3_aa for the heavy and light chains
    if 'IGH_junction_aa' not in cols and 'IGH_cdr3_aa' not in cols:
        raise ValueError("Input must contain either 'IGH_junction_aa' or 'IGH_cdr3_aa'")
    if 'IGL_junction_aa' not in cols and 'IGL_cdr3_aa' not in cols:
        raise ValueError("Input must contain either 'IGL_junction_aa' or 'IGL_cdr3_aa'")
 
    # Reject ANY NaN in required columns
    if df[req_fields].isna().any().any():
        bad = df[req_fields].isna().any()
        bad_cols = bad[bad].index.tolist()
        raise ValueError(f"NaN values found in required columns: {', '.join(bad_cols)}")
 
    # Reject empty strings in required string columns
    for col in ['IGH_v_call', 'IGH_j_call','IGL_v_call', 'IGL_j_call']:
        if df[col].astype('string').str.strip().eq('').any():
            raise ValueError(f"Column '{col}' contains empty strings")
 
    
    print('All required fields are present and valid in the input data')
    
    
# Load data
#===========
# Base directory
pth = ""
"""
IN_FILE_PATH: Path to the input file.

Supported input formats:
    - AIRR (Adaptive Immune Receptor Repertoire) TSV file
    - Excel file (.xlsx or .xls)
    - TSV file containing the required columns for BCR heavy chain listed below
"""

IN_FILE_PATH = Path(pth, "BCR_paired_chain_input.tsv") # sample file
# out_file_path is a tsv 
OUT_FILE_PATH = Path(pth, "BCR_paired_chain_clonotypes.tsv")

df = load_input_data(
    in_file_path=IN_FILE_PATH,
    out_file_path=OUT_FILE_PATH
)

# Validate input
check_input(df)

All required fields are present and valid in the input data


In [3]:
# Functions for Network-Based BCR Clonotype Identification

def sim_score(a,b):
    """ Calculate pairwise identity score for CDR3 AA sequences of same length """
    return sum([1 if a[i]==b[i] else 0 for i in range(len(a))])/len(a) 


def find_families(cdr3s):
    """ Take a list of cdr3s and group into families with 80% sequence identity
      Using itertools.combinations  https://docs.python.org/3/library/itertools.html#itertools.combinations """
    edges = []
    for i,j in itertools.combinations(range(len(cdr3s)),2): 
        if sim_score(cdr3s[i], cdr3s[j]) >= 0.80:
            edges.append((i,j))
    G = nx.Graph()
    G.add_nodes_from(range(len(cdr3s)))
    G.add_edges_from(edges)
    families = []
    for c in nx.connected_components(G):
        families.append([cdr3s[i] for i in c])
    return families


def same_VH_cdr3_family(df):
    """ Within each group with same HC  same junction length, use networkx to find those with 80% identity.
        ... for heavy chain cdr3 AA ... """
    hc_jcts = df['IGH_cdr3_aa'].unique()
    cdr3_igh_families = []
    for cdr3_family in find_families(hc_jcts):
        cdr3_igh_families.append (df[df['IGH_cdr3_aa'].isin(cdr3_family)])
    return cdr3_igh_families

def same_VL_cdr3_family(df):
    """ Within each group with same LC  same junction length, use networkx to find those with 80% identity.
        ... for light chain cdr3 AA... """
    lc_jcts = df['IGL_cdr3_aa'].unique()
    cdr3_igl_families = []
    for cdr3_family in find_families(lc_jcts):
        cdr3_igl_families.append (df[df['IGL_cdr3_aa'].isin(cdr3_family)])
    return cdr3_igl_families





In [4]:
# Data Processing 


df['VH_v_call']=df['IGH_v_call'].apply(lambda x: x.split('*')[0] if x else None)
df['VH_j_call']=df['IGH_j_call'].apply(lambda x: x.split('*')[0] if x else None)
df['VL_v_call']=df['IGL_v_call'].apply(lambda x: x.split('*')[0] if x else None)
df['VL_j_call']=df['IGL_j_call'].apply(lambda x: x.split('*')[0] if x else None)


#combine V and J gene annotations
df['VH_vj']=df['VH_v_call']+"_"+df['VH_j_call']
df['VL_vj']=df['VL_v_call']+"_"+df['VL_j_call'] 

# If cdr3_aa is not available, derive CDR3 amino acid sequences from
# junction_aa by removing the conserved N-terminal C and the conserved
# C-terminal residue (typically W or F).
if not "IGH_cdr3_aa" in df.columns:
    df["IGH_cdr3_aa"] = df["IGH_junction_aa"].astype(str).str[1:-1]
if not "IGL_cdr3_aa" in df.columns:
    df["IGL_cdr3_aa"] = df["IGL_junction_aa"].astype(str).str[1:-1]

# Calculate the length of CDR3 amino acid sequences
df["IGH_cdr3_aa_length"]=df["IGH_cdr3_aa"].astype(str).str.len() 
df["IGL_cdr3_aa_length"]=df["IGL_cdr3_aa"].astype(str).str.len() 

clonotype_no=0

# Group BCR heavy chain sequences by V gene, J gene, and junction length
grouped_VH= df.groupby(["VH_vj", "IGH_cdr3_aa_length"])
# Convert each groups into a DataFrame and store them in a dictionary
grouped_VH_df = {name: group.reset_index(drop=True) for name, group in grouped_VH}

#Assign clonotypes using a network-based clustering approach
for name, group_df in grouped_VH_df.items():
    
    for cdr3_igh_family in same_VH_cdr3_family(group_df):  
        ## Group heavy-chain sequences sharing the same light-chain V gene, J gene, and junction length
        grouped_VL = cdr3_igh_family.groupby(["VL_vj", "IGL_cdr3_aa_length"])
        grouped_VL_df = {name: group.reset_index(drop=True) for name, group in grouped_VL}
        for VL_name,group_df_VL in grouped_VL_df.items():
            for cdr3_igl_family in same_VL_cdr3_family(group_df_VL):
            
                family_name="clonotype_%d" % clonotype_no
                df_group=cdr3_igl_family.copy()
                df_group['clonotype_family']=family_name
            
                if clonotype_no == 0: # Include header
                    df_group.to_csv(OUT_FILE_PATH, sep='\t',mode='w', index=False, header=True)  
                else:
                    df_group.to_csv(OUT_FILE_PATH, sep='\t',mode='a', index=False, header=False) # Exclude header
            
                clonotype_no += 1
